# 15-тарау. Үлкен тілдік модельдер қалай жасалады?

*«Машина қалай үйренеді? 2-том» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 15.1. Біздің GPT пен үлкен модельдердің айырмасы

## 15.2. 1-кезең. Алдын ала оқыту

In [ ]:
models = {"Біздің GPT":      (0.81e6, 2500 * 32 * 64),
          "GPT-3":           (175e9, 300e9),
          "Chinchilla":      (70e9, 1.4e12),
          "Llama 3.1 405B":  (405e9, 15.6e12)}
print("модель            параметрлер  токендер  токен/параметр  есептеу")
for name, (N, D) in models.items():
    C = 6 * N * D
    print(f"{name:16s} {N:12.1e} {D:9.1e} {D / N:15.1f} {C:8.1e}")
ratio = 6 * 405e9 * 15.6e12 / (6 * 0.81e6 * 2500 * 32 * 64)
print(f"Llama 3.1 біздің модельден {ratio:.1e} есе көп есептеу жұмсады")

## 15.3. 2-кезең. Нұсқаулық бойынша баптау

In [ ]:
dialog = [
    {"role": "system",
     "content": "Сен — оқушыларға көмектесетін көмекшісің."},
    {"role": "user", "content": "Фотосинтез деген не?"},
    {"role": "assistant",
     "content": "Өсімдіктердің күн сәулесімен қоректік зат жасауы."},
]

def to_text(dialog):
    return "".join(f"<|{m['role']}|>\n{m['content']}\n<|end|>\n"
                   for m in dialog)

print(to_text(dialog))

## 15.4. LoRA: модельдің аз бөлігін баптау

In [ ]:
import urllib.request
import torch
from torch import nn
import torch.nn.functional as F

RAW = ("https://raw.githubusercontent.com/amirtemir1996n3-dotcom/"
       "mashina-qalai-uirenedi/main/")
for path in ["data/kitap.txt", "data/qa.txt", "vol2/gpt.py",
             "vol2/gpt_kitap.pt"]:
    urllib.request.urlretrieve(RAW + path, path.split("/")[-1])
from gpt import GPT, GPTConfig

chars = sorted(set(open("kitap.txt", encoding="utf-8").read()))
stoi = {ch: i for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(chars[i] for i in ids)

qa = open("qa.txt", encoding="utf-8").read().strip().split("\n\n")
print(len(qa), "сұрақ-жауап")
print(qa[5])
train_text = "\n\n".join(qa[:-8]) + "\n\n"
test_text = "\n\n".join(qa[-8:]) + "\n\n"

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, r=4, alpha=8):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale

def load_base():
    m = GPT(GPTConfig(vocab_size=len(chars), block_size=64))
    m.load_state_dict(torch.load("gpt_kitap.pt"))
    return m

torch.manual_seed(0)
model = load_base()
for p in model.parameters():
    p.requires_grad = False                      # бәрін қатырамыз
for block in model.blocks:                       # назар қабаттарына LoRA
    block.attn.qkv = LoRALinear(block.attn.qkv)
    block.attn.proj = LoRALinear(block.attn.proj)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"оқытылатыны: {trainable} / {total} ({trainable / total:.1%})")

In [ ]:
train_ids = torch.tensor(encode(train_text))
test_ids = torch.tensor(encode(test_text))

def batch(ids, n=16, T=64):
    ix = torch.randint(len(ids) - T - 1, (n,))
    x = torch.stack([ids[i:i + T] for i in ix])
    y = torch.stack([ids[i + 1:i + T + 1] for i in ix])
    return x, y

@torch.no_grad()
def test_loss(m):
    m.eval()
    torch.manual_seed(123)
    loss = sum(m(*batch(test_ids))[1].item() for _ in range(20)) / 20
    m.train()
    return loss

print(f"баптауға дейін: {test_loss(model):.3f}")
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                        lr=3e-3)
torch.manual_seed(0)
for step in range(1, 601):
    _, loss = model(*batch(train_ids))
    opt.zero_grad()
    loss.backward()
    opt.step()
print(f"баптаудан кейін: {test_loss(model):.3f}")

In [ ]:
@torch.no_grad()
def answer(m, question, n=160):
    m.eval()
    ids = encode(f"Сұрақ: {question}\nЖауап:")
    for _ in range(n):
        logits = m(torch.tensor([ids[-64:]]))[0][0, -1]
        probs = torch.softmax(logits / 0.7, dim=-1)
        ids.append(int(torch.multinomial(probs, 1)))
        if decode(ids).endswith("\n\n"):
            break
    return decode(ids).strip()

base = load_base()
for name, m in [("бастапқы модель", base), ("LoRA-мен бапталған", model)]:
    torch.manual_seed(5)
    print(f"--- {name} ---")
    print(answer(m, "Тензор деген не?"))

In [ ]:
lora = model.blocks[0].attn.qkv
merged = nn.Linear(lora.base.in_features, lora.base.out_features)
with torch.no_grad():
    merged.weight.copy_(lora.base.weight + lora.scale * lora.B @ lora.A)
    merged.bias.copy_(lora.base.bias)
x = torch.randn(2, 5, 128)
print(torch.allclose(merged(x), lora(x), atol=1e-5))

## 15.5. 3-кезең. Адамның бағасынан үйрену

In [ ]:
def dpo_loss(pol_w, pol_l, ref_w, ref_l, beta=0.1):
    margin = beta * ((pol_w - ref_w) - (pol_l - ref_l))
    return -F.logsigmoid(torch.tensor(margin)).item()

# лог-ықтималдықтар: жақсы жауап, нашар жауап
print(f"модель жақсы жауапқа ауысты: {dpo_loss(-12, -15, -13, -14):.3f}")
print(f"өзгеріс жоқ:                 {dpo_loss(-13, -14, -13, -14):.3f}")
print(f"модель нашар жауапқа ауысты: {dpo_loss(-14, -13, -13, -14):.3f}")

## 15.6. Ойланатын модельдер